In [1]:
import pandas as pd
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from openai import OpenAI
from pydantic import BaseModel
import json

# Load API key
with open("key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)


In [2]:
import re

# Arabic tashkeel unicode range
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return text
    return re.sub(TASHKEEL_REGEX, '', text)


In [3]:
# 2. Data Loading (BOTH DATASETS)
# ===============================
print("Loading Datasets...")

# Load Majaz (Positive)
df_pos = pd.read_csv("majaz_dataset.csv")
df_pos = df_pos[["البيت_الكامل"]].copy()
df_pos["label"] = 1

# Load Non-Majaz (Negative)
df_neg = pd.read_csv("non_majaz_random_sample.csv")
df_neg = df_neg[["البيت_الكامل"]].copy()
df_neg["label"] = 0

# Combine
df = pd.concat([df_pos, df_neg], ignore_index=True)
TEXT_COL = "البيت_الكامل"

# Clean
df = df.dropna(subset=[TEXT_COL])
df[TEXT_COL] = df[TEXT_COL].apply(remove_tashkeel)

# Use ALL Data
X_test = df[TEXT_COL].values
y_test = df["label"].values

print(f"Total verses to test: {len(X_test)}")
print(f"Majaz count: {sum(y_test)}")
print(f"Non-Majaz count: {len(y_test) - sum(y_test)}")
print(X_test[-5:])
print(y_test[-5:])

Loading Datasets...
Total verses to test: 1594
Majaz count: 836
Non-Majaz count: 758
['جري الجنان لا أهال من الردى إذا ما جعلت السيف من عن شماليا'
 'ووجه له ديباجة قرشية به تكشف البلوى ويستنزل القطر'
 'رأيت الحر يجتنب المخازي ويحميه عن الغدر الوفاء'
 'حتى أفجعهم بإخوتهم وأسوق نسوتهم بنسوتيه'
 'أين جوب البلاد شرقا وغربا واعتساف السهول والأجبال']
[0 0 0 0 0]


In [4]:
# 3. Define Clean Zero-Shot Prompt
# ================================
class VerseClassification(BaseModel):
    id: int
    reasoning: str 
    is_majaz: int

class BatchResponse(BaseModel):
    results: list[VerseClassification]

SYSTEM_PROMPT = """
أنت ناقد خبير بلاغي ومتخصص في عنصر المجاز
سأعطيك بيت شعر، افحص اذا كان فيه العنصر البلاغي المجاز ام لا

المطلوب:
حلل البيت، ثم قرر: 1 (مجاز) أو 0 (ليس مجازاً).
"""

def get_predictions_batch(verses, start_index):
    formatted_verses = "\n".join([f"ID {i+start_index}: {v}" for i, v in enumerate(verses)])

    user_content = f"""
    الأبيات:
    {formatted_verses}
    """

    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-5.2-2025-12-11", # Strongest available model for reasoning
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
            ],
            response_format=BatchResponse,
            temperature=0.0,
        )
        return completion.choices[0].message.parsed.results
    except Exception as e:
        print(f"Error: {e}")
        return []

In [5]:
# 4. Execution
# ============
BATCH_SIZE = 1 
predictions_map = {}

print("Running Zero-Shot Test...")

for i in tqdm(range(0, len(X_test), BATCH_SIZE)):
    batch_verses = X_test[i : i + BATCH_SIZE]
    results = get_predictions_batch(batch_verses, start_index=i)
    
    for res in results:
        predictions_map[res.id] = {"pred": res.is_majaz, "reason": res.reasoning}

Running Zero-Shot Test...


100%|██████████| 1594/1594 [1:50:18<00:00,  4.15s/it] 


In [6]:
# ================================
# 5. FIXED RESULTS SECTION
# ================================
final_preds = []
final_reasons = []

for i in range(len(X_test)):
    if i in predictions_map:
        final_preds.append(predictions_map[i]["pred"])
        final_reasons.append(predictions_map[i]["reason"])
    else:
        final_preds.append(0) 
        final_reasons.append("Error")

# 1. Calculate Accuracy (Overall Performance)
acc = accuracy_score(y_test, final_preds)
print(f"\nAccuracy: {acc:.2%}")

# 2. Detailed Report
print("\nClassification Report:")
print(classification_report(y_test, final_preds, target_names=["Non-Majaz (0)", "Majaz (1)"]))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, final_preds)
print(cm)

# Save Results
results_df = pd.DataFrame({
    "verse": X_test,
    "y_true": y_test,
    "y_pred": final_preds,
    "reasoning": final_reasons
})

# 3. CORRECTLY Identify Errors
# False Negatives: It WAS Majaz (1), but model said Non-Majaz (0)
false_negatives = results_df[(results_df["y_true"] == 1) & (results_df["y_pred"] == 0)]

# False Positives: It WAS Non-Majaz (0), but model said Majaz (1)
false_positives = results_df[(results_df["y_true"] == 0) & (results_df["y_pred"] == 1)]

print(f"\nMissed Majaz (False Negatives): {len(false_negatives)}")
print(f"Wrongly Accused as Majaz (False Positives): {len(false_positives)}")

if not false_negatives.empty:
    print("\nExample of Missed Majaz:")
    print(false_negatives[["verse", "reasoning"]].head(2))
    false_negatives.to_csv("errors_false_negatives.csv", index=False, encoding="utf-8-sig")

if not false_positives.empty:
    print("\nExample of False Positive:")
    print(false_positives[["verse", "reasoning"]].head(2))
    false_positives.to_csv("errors_false_positives.csv", index=False, encoding="utf-8-sig")


Accuracy: 54.33%

Classification Report:
               precision    recall  f1-score   support

Non-Majaz (0)       0.58      0.14      0.23       758
    Majaz (1)       0.54      0.91      0.68       836

     accuracy                           0.54      1594
    macro avg       0.56      0.52      0.45      1594
 weighted avg       0.56      0.54      0.46      1594


Confusion Matrix:
[[107 651]
 [ 77 759]]

Missed Majaz (False Negatives): 77
Wrongly Accused as Majaz (False Positives): 651

Example of Missed Majaz:
                                                verse  \
10  وأشقى بلاد الله ما الروم أهلها بهذا وما فيها ل...   
11  فلم يبق إلا من حماها من الظبا لمى شفتيها والثد...   

                                            reasoning  
10  في البيت حكمٌ تقريريّ: يصف الشاعر بلادَ الروم ...  
11  في البيت تصويرٌ للمرأة/المحبوبة بذكر صفات حسّي...  

Example of False Positive:
                                                 verse  \
836  تعاللتها وليس طبي بدرها وكيف التماس الدر و

In [7]:
# ==========================================
# SAVE RESULTS TO CSV
# ==========================================
# 1. Create the DataFrame
output_df = pd.DataFrame({
    "Verse_Text": X_test,           # The Arabic text
    "True_Label": y_test,           # 0 or 1 (Ground Truth)
    "Predicted_Label": final_preds, # 0 or 1 (Model Prediction)
    "Match": [x == y for x, y in zip(y_test, final_preds)], # True if correct, False if wrong
    "Model_Reasoning": final_reasons # The explanation given by GPT
})

# 2. Save to CSV
# 'utf-8-sig' is CRITICAL for Arabic to open correctly in Excel
file_name = "gpt_majaz_final_results.csv"
output_df.to_csv(file_name, index=False, encoding="utf-8-sig")

print(f"✅ Successfully saved results to: {file_name}")

# 3. Preview the first 5 rows
print("\nPreview:")
print(output_df.head())

✅ Successfully saved results to: gpt_majaz_final_results.csv

Preview:
                                          Verse_Text  True_Label  \
0  مررت على دار الحبيب فحمحمت جوادي وهل تشجو الجي...           1   
1  وتسعدني في غمرة بعد غمرة سبوح لها منها عليها ش...           1   
2   محرمة أكفال خيلي على القنا محللة لباتها والقلائد           1   
3  ولكن إذا لم يحمل القلب كفه على حالة لم يحمل ال...           1   
4  له من كريم الطبع في الحرب منتض ومن عادة الإحسا...           1   

   Predicted_Label  Match                                    Model_Reasoning  
0                1   True  في البيت مجاز واضح في إسناد فعلٍ من أفعال الإن...  
1                1   True  في البيت مجاز؛ إذ أُسنِد الفعل «تسعدني» إلى «س...  
2                1   True  في البيت مجاز واضح في قوله: «محرَّمة أكفالُ خي...  
3                1   True  في البيت إسنادٌ مجازيّ (مجاز عقلي)؛ إذ جُعل «ا...  
4                1   True  في البيت استعمالٌ للمجاز في قوله: «في الحرب مُ...  
